# Gold Price Prediction
## Time Series Forecasting using ML and DL

This notebook contains:
- Linear Regression (Machine Learning)
- LSTM (Deep Learning)
- Data preprocessing
- Time series forecasting
- Model evaluation and visualization

In [ ]:
# Install required libraries (Run if needed)
# !pip install pandas numpy scikit-learn tensorflow matplotlib

In [ ]:
# =========================
# IMPORT LIBRARIES
# =========================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.preprocessing import MinMaxScaler
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, LSTM

In [ ]:
# =========================
# LOAD DATASET
# =========================

df = pd.read_csv(
    "gold_price.csv",
    engine='python',
    on_bad_lines='skip'
)

print(df.head())
print(df.columns)

## Note
Update the column names according to your dataset.

Example:
- Date column: `Date`
- Price column: `Price`

In [ ]:
# =========================
# SELECT REQUIRED COLUMNS
# =========================

date_column = "Date"
price_column = "Price"

# Keep only required columns
df = df[[date_column, price_column]]

# Remove missing values
df.dropna(inplace=True)

print(df.head())

In [ ]:
# =========================
# DATE PROCESSING
# =========================

df[date_column] = pd.to_datetime(df[date_column])

# Sort by date
df = df.sort_values(by=date_column)

# Reset index
df.reset_index(drop=True, inplace=True)

print(df.head())

In [ ]:
# =========================
# VISUALIZE GOLD PRICES
# =========================

plt.figure(figsize=(12,6))

plt.plot(df[date_column], df[price_column])

plt.title("Gold Price Over Time")
plt.xlabel("Date")
plt.ylabel("Gold Price")

plt.show()

In [ ]:
# =========================
# NORMALIZATION
# =========================

scaler = MinMaxScaler(feature_range=(0,1))

scaled_data = scaler.fit_transform(
    df[[price_column]]
)

print(scaled_data[:5])

In [ ]:
# =========================
# CREATE TIME SERIES DATA
# =========================

def create_dataset(data, time_step=10):

    X = []
    y = []

    for i in range(len(data) - time_step - 1):

        X.append(data[i:(i + time_step), 0])
        y.append(data[i + time_step, 0])

    return np.array(X), np.array(y)

time_step = 10

X, y = create_dataset(scaled_data, time_step)

print(X.shape)
print(y.shape)

In [ ]:
# =========================
# TRAIN TEST SPLIT
# =========================

train_size = int(len(X) * 0.8)

X_train = X[:train_size]
X_test = X[train_size:]

y_train = y[:train_size]
y_test = y[train_size:]

print(X_train.shape)
print(X_test.shape)

## Machine Learning Technique - Linear Regression

In [ ]:
# =========================
# LINEAR REGRESSION MODEL
# =========================

lr_model = LinearRegression()

# Train model
lr_model.fit(X_train, y_train)

# Predictions
lr_predictions = lr_model.predict(X_test)

# Evaluation
mse = mean_squared_error(y_test, lr_predictions)
mae = mean_absolute_error(y_test, lr_predictions)

print("Mean Squared Error:", mse)
print("Mean Absolute Error:", mae)

## Deep Learning Technique - LSTM

In [ ]:
# =========================
# RESHAPE DATA FOR LSTM
# =========================

X_train_lstm = X_train.reshape(
    X_train.shape[0],
    X_train.shape[1],
    1
)

X_test_lstm = X_test.reshape(
    X_test.shape[0],
    X_test.shape[1],
    1
)

print(X_train_lstm.shape)

In [ ]:
# =========================
# BUILD LSTM MODEL
# =========================

lstm_model = Sequential()

lstm_model.add(
    LSTM(
        50,
        return_sequences=True,
        input_shape=(time_step, 1)
    )
)

lstm_model.add(LSTM(50))

lstm_model.add(Dense(1))

# Compile model
lstm_model.compile(
    optimizer='adam',
    loss='mean_squared_error'
)

lstm_model.summary()

In [ ]:
# =========================
# TRAIN LSTM MODEL
# =========================

history = lstm_model.fit(
    X_train_lstm,
    y_train,
    epochs=20,
    batch_size=32,
    validation_data=(X_test_lstm, y_test)
)

In [ ]:
# =========================
# LSTM PREDICTIONS
# =========================

lstm_predictions = lstm_model.predict(
    X_test_lstm
)

# Inverse transform
lstm_predictions = scaler.inverse_transform(
    lstm_predictions
)

actual_prices = scaler.inverse_transform(
    y_test.reshape(-1,1)
)

print(lstm_predictions[:5])

In [ ]:
# =========================
# VISUALIZE PREDICTIONS
# =========================

plt.figure(figsize=(12,6))

plt.plot(actual_prices, label='Actual Prices')

plt.plot(lstm_predictions, label='Predicted Prices')

plt.title("Gold Price Prediction")

plt.xlabel("Time")
plt.ylabel("Gold Price")

plt.legend()

plt.show()

## Conclusion

- Linear Regression provides a basic forecasting model.
- LSTM performs better for sequential and time-series data.
- This project demonstrates ML and DL approaches for gold price forecasting.